# Build sum-of-product vibrational KEO for formaldehyde ($\mathrm{H_2CO}$) using valence coordinates

In [1]:
import jax
import numpy as np
from jax import numpy as jnp
from scipy import constants

from gvib import Gvib
from keo_jax import batch_Gmat, com
from revad import dihedral, dot, norm
import functools

jax.config.update("jax_enable_x64", True)

G_to_invcm = (
    constants.value("Planck constant")
    * constants.value("Avogadro constant")
    * 1e16
    / (4.0 * np.pi**2 * constants.value("speed of light in vacuum"))
    * 1e5
)

In [2]:
def com_sop(masses: np.ndarray):
    def wrapper(internal_to_cartesian):
        @functools.wraps(internal_to_cartesian)
        def wrapper_com(*args, **kwargs):
            xyz = internal_to_cartesian(*args, **kwargs)
            assert len(xyz) == len(masses), (
                "The number of elements in 'masses' must match the leading dimension of the array "
                "returned by the 'internal_to_cartesian' function"
            )
            masses_ = np.asarray(masses)
            com = masses_ @ xyz / np.sum(masses_)
            return xyz - com[None, :]

        return wrapper_com

    return wrapper



Define the internal-to-Cartesian and Cartesian-to-internal coordinate transformations for $\mathrm{H_2CO}$.

The internal coordinates are defined as:

1. $r_{\mathrm{CO}}$ — C–O bond length,
2. $r_{\mathrm{CH_1}}$ — C–H1 bond length,
3. $r_{\mathrm{CH_2}}$ — C–H2 bond length,
4. $\alpha_{\mathrm{H_1CO}}$ — H1–C–O bond angle,
5. $\alpha_{\mathrm{H_2CO}}$ — H2–C–O bond angle,
6. $\Delta\tau = \tau + \pi$, where $\tau$ is the book angle (dihedral angle) between the (H1CO) and (H2CO) planes.

The atom ordering in Cartesian coordinates is:

$$
\mathrm{C},\ \mathrm{O},\ \mathrm{H_1},\ \mathrm{H_2}
$$

In [3]:
masses = np.array([12.0, 15.99491463, 1.00782505, 1.00782505])  # masses of C, O, H, H

ncoo = 6


# @com_sop(masses)
def internal_to_cartesian(q):
    rCO, rCH1, rCH2, aOCH1, aOCH2, dtau = q
    tau = np.pi + dtau
    xyz = [
        [0.0, 0.0, 0.0],
        [0.0, 0.0, rCO],
        [
            rCH1 * np.sin(aOCH1) * np.cos(tau * 0.5),
            -rCH1 * np.sin(aOCH1) * np.sin(tau * 0.5),
            rCH1 * np.cos(aOCH1),
        ],
        [
            rCH2 * np.sin(aOCH2) * np.cos(tau * 0.5),
            rCH2 * np.sin(aOCH2) * np.sin(tau * 0.5),
            rCH2 * np.cos(aOCH2),
        ],
    ]
    return np.array(xyz, dtype=object)


def cartesian_to_internal(xyz):
    C, O, H1, H2 = xyz

    CO = O - C
    CH1 = H1 - C
    CH2 = H2 - C

    rCO = norm(CO)
    rCH1 = norm(CH1)
    rCH2 = norm(CH2)

    aOCH1 = np.arccos(dot(CO, CH1) / (rCO * rCH1))
    aOCH2 = np.arccos(dot(CO, CH2) / (rCO * rCH2))

    tau = dihedral(H1, C, O, H2)
    dtau = tau + np.pi

    return np.array([rCO, rCH1, rCH2, aOCH1, aOCH2, dtau], dtype=object)

Define one-dimensional grids for the internal coordinate representation and construct the vibrational G-matrix in sum-of-products form

In [4]:
grids = [
    np.linspace(0.8, 1.2, 5),  # r_{CO}
    np.linspace(0.8, 1.2, 5),  # r_{CH1}
    np.linspace(0.8, 1.2, 5),  # r_{CH2}
    np.linspace(60 * np.pi / 180, 160 * np.pi / 180, 5),  # alpha_{H1CO}
    np.linspace(60 * np.pi / 180, 160 * np.pi / 180, 5),  # alpha_{H2CO}
    np.linspace(-60 * np.pi / 180, 60 * np.pi / 180, 5),  # dtau
]


gvib = Gvib(grids, masses, internal_to_cartesian, cartesian_to_internal)
gvib_sop, grovib_sop, grot_sop, data = gvib.run(return_rot=True, return_svec=True)

print("Number of terms in Gvib_{i,j}\n i  j no.terms")

print('-------')
for i in range(ncoo):
    for j in range(i, ncoo):
        print(f"{i:2d} {j:2d} {gvib_sop[i, j].nterms:6d}")

print('-------')
for i in range(3):
    for j in range(ncoo):
        print(f"{i:2d} {j:2d} {grovib_sop[i, j].nterms:6d}")

print('-------')
for i in range(3):
    for j in range(i, 3):
        print(f"{i:2d} {j:2d} {grot_sop[i, j].nterms:6d}")

Number of terms in Gvib_{i,j}
 i  j no.terms
-------
 0  0      1
 0  1      1
 0  2      1
 0  3      2
 0  4      2
 0  5      1
 1  1      1
 1  2      3
 1  3      4
 1  4      6
 1  5      2
 2  2      1
 2  3      6
 2  4      4
 2  5      2
 3  3      7
 3  4     12
 3  5      4
 4  4      7
 4  5      4
 5  5     14
-------
 0  0      1
 0  1      1
 0  2      1
 0  3      2
 0  4      2
 0  5      4
 1  0      1
 1  1      1
 1  2      1
 1  3      2
 1  4      2
 1  5      4
 2  0      1
 2  1      2
 2  2      2
 2  3      4
 2  4      4
 2  5      6
-------
 0  0      1
 0  1      1
 0  2      4
 1  1      1
 1  2      4
 2  2     14


Evaluate the vibrational G-matrix numerically on the multidimensional grid formed by the Cartesian product of the one-dimensional grids used to construct the sum-of-products representation

In [5]:
@com(masses)
def internal_to_cartesian_jax(q):
    rCO, rCH1, rCH2, aOCH1, aOCH2, dtau = q
    tau = jnp.pi + dtau
    xyz = [
        [0.0, 0.0, 0.0],
        [0.0, 0.0, rCO],
        [
            rCH1 * jnp.sin(aOCH1) * jnp.cos(tau * 0.5),
            -rCH1 * jnp.sin(aOCH1) * jnp.sin(tau * 0.5),
            rCH1 * jnp.cos(aOCH1),
        ],
        [
            rCH2 * jnp.sin(aOCH2) * jnp.cos(tau * 0.5),
            rCH2 * jnp.sin(aOCH2) * jnp.sin(tau * 0.5),
            rCH2 * jnp.cos(aOCH2),
        ],
    ]
    return jnp.array(xyz, dtype=jnp.float64)


grid_ndim = np.stack(np.meshgrid(*grids, indexing="ij"), axis=-1)
grid_ndim = grid_ndim.reshape(-1, len(grids))

gmat = batch_Gmat(grid_ndim, masses, internal_to_cartesian_jax)

gvib_ndim = gmat[:, :ncoo, :ncoo]
grovib_ndim = gmat[:, ncoo : ncoo + 3, :ncoo]
grot_ndim = gmat[:, ncoo : ncoo + 3, ncoo : ncoo + 3]

print("gvib_ndmm.shape = ", gvib_ndim.shape)

gvib_ndmm.shape =  (15625, 6, 6)


Calculate point-wise differences between the sum-of-products and numerical vibrational G-matrix

In [6]:
# number of multidimensional grid points
ngrid = grid_ndim.shape[0]

# evaluate the sum-of-products G-matrix on the full direct-product grid
gvib_sop_eval = np.zeros((ngrid, ncoo, ncoo), dtype=np.float64)
grot_sop_eval = np.zeros((ngrid, 3, 3), dtype=np.float64)
grovib_sop_eval = np.zeros((ngrid, 3, ncoo), dtype=np.float64)

# map each multidimensional grid point to its 1D grid indices
grid_indices = np.indices([len(grid) for grid in grids]).reshape(len(grids), -1).T

for ipoint, inds in enumerate(grid_indices):
    for i in range(ncoo):
        for j in range(ncoo):
            sop = gvib_sop[i, j]

            value = 0.0

            for iterm in range(sop.nterms):
                term = 1.0

                for k in range(ncoo):
                    term *= sop.terms[iterm][k][inds[k]]

                value += term

            gvib_sop_eval[ipoint, i, j] = value * G_to_invcm

for ipoint, inds in enumerate(grid_indices):
    for i in range(3):
        for j in range(3):
            sop = grot_sop[i, j]

            value = 0.0

            for iterm in range(sop.nterms):
                term = 1.0

                for k in range(ncoo):
                    term *= sop.terms[iterm][k][inds[k]]

                value += term

            grot_sop_eval[ipoint, i, j] = value * G_to_invcm

for ipoint, inds in enumerate(grid_indices):
    for i in range(3):
        for j in range(ncoo):
            sop = grovib_sop[i, j]

            value = 0.0

            for iterm in range(sop.nterms):
                term = 1.0

                for k in range(ncoo):
                    term *= sop.terms[iterm][k][inds[k]]

                value += term

            grovib_sop_eval[ipoint, i, j] = value * G_to_invcm

In [7]:
mask = np.abs(grovib_sop_eval) > 1e-8

ratio = grovib_ndim[mask] / grovib_sop_eval[mask]

print("ratio min :", ratio.min())
print("ratio max :", ratio.max())
print("ratio mean:", ratio.mean())
print("ratio std :", ratio.std())

ratio min : 0.9999999919995815
ratio max : 1.0000000008191618
ratio mean: 0.9999999999998965
ratio std : 4.8512117976952483e-11


In [8]:
# differences

diff_vib = gvib_sop_eval - gvib_ndim
diff_rovib = grovib_sop_eval - grovib_ndim
diff_rot = grot_sop_eval - grot_ndim

abs_diff_vib = np.abs(diff_vib)
abs_diff_rovib = np.abs(diff_rovib)
abs_diff_rot = np.abs(diff_rot)

print("Maximum absolute difference:", np.max(abs_diff_vib))
print("Mean absolute difference:", np.mean(abs_diff_vib))
print("RMS difference:", np.sqrt(np.mean(diff_vib**2)))

print("\nMaximum absolute difference:", np.max(abs_diff_rovib))
print("Mean absolute difference:", np.mean(abs_diff_rovib))
print("RMS difference:", np.sqrt(np.mean(diff_rovib**2)))

print("\nMaximum absolute difference:", np.max(abs_diff_rot))
print("Mean absolute difference:", np.mean(abs_diff_rot))
print("RMS difference:", np.sqrt(np.mean(diff_rot**2)))

Maximum absolute difference: 3.296918293926865e-12
Mean absolute difference: 6.818715827401518e-15
RMS difference: 4.1351575754844234e-14

Maximum absolute difference: 8.165614253420022e-11
Mean absolute difference: 1.3311692201975782e-12
RMS difference: 5.90655581279627e-12

Maximum absolute difference: 5.644551492878236e-11
Mean absolute difference: 1.872460571834662e-12
RMS difference: 5.3154779207332945e-12
